# Automata and Formal Languages

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 07.03 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/07_Discrete_Mathematics/04_automata_and_formal_languages.ipynb)

---

## 🎯 Learning Objective

Learn finite automata, regular expressions, and context-free grammars, and place them on the Chomsky hierarchy.

---

## 📐 Theory

A **formal language** is simply a set of strings over some alphabet $\Sigma$ — for example, the
set of all strings over $\{0,1\}$ with an even number of 1's, or the set of all syntactically
valid Python programs. Formal language theory studies *how hard* different languages are to
describe and recognize, and the answer turns out to organize neatly into a hierarchy of
increasing power.

### Deterministic finite automata (DFA)

A **deterministic finite automaton** is the simplest string-recognizing machine: a finite set
of **states** $Q$, an alphabet $\Sigma$, a **transition function** $\delta:Q\times\Sigma\to Q$
(read one symbol, move to exactly one next state), a **start state** $q_0$, and a set of
**accepting states** $F\subseteq Q$. Feed a DFA a string one symbol at a time; it accepts the
string if and only if it ends in an accepting state. Crucially, a DFA has **no memory beyond
its current state** — the entire history of the input is compressed into "which one of finitely
many states am I in right now." This constraint is exactly what makes DFAs so limited, and
exactly what makes them so cheap: recognizing a string of length $n$ costs $O(n)$ time and
$O(1)$ space, regardless of the string.

### Nondeterministic finite automata (NFA) and regular expressions

An **NFA** relaxes the transition function to allow zero, one, or many next states for a given
(state, symbol) pair — informally, "guess" which transition to take, and accept if *some* choice
sequence leads to acceptance. NFAs and DFAs recognize exactly the same class of languages (any
NFA can be mechanically converted to an equivalent DFA, via the *subset construction*, at the
cost of potentially exponentially more states) — this class is called the **regular languages**,
and **regular expressions** are just a compact algebraic notation (concatenation, alternation
`|`, repetition `*`) for describing them. Every regex engine is, underneath, compiling a pattern
into something equivalent to an NFA or DFA.

### Context-free grammars and the Chomsky hierarchy

Some languages provably cannot be recognized by any DFA. The classic example is
**balanced parentheses** of unbounded depth: a DFA has finitely many states, so it can track
nesting depth only up to some fixed maximum before it runs out of distinct states to represent
"how deep am I" — beyond that bound, two different depths must reuse the same state, and the
machine loses the information it needs to correctly match the final closing parenthesis. A
**context-free grammar (CFG)** fixes this with *recursive production rules*, e.g. $S \to
(S)S \mid \varepsilon$ (a balanced string is either empty, or a balanced string wrapped in
parentheses, followed by another balanced string) — recognizing a CFG's language requires a
**stack** (unbounded memory), not just a finite state.

The **Chomsky hierarchy** ranks four language classes by the machinery needed to recognize them:

| Grammar type | Recognizing machine | Example |
|---|---|---|
| Type 3: Regular | Finite automaton (DFA/NFA), no extra memory | Fixed-pattern strings, bounded-depth nesting |
| Type 2: Context-free | Pushdown automaton (finite states + a stack) | Balanced parentheses, arithmetic expressions, most programming-language syntax |
| Type 1: Context-sensitive | Linear-bounded automaton | Natural-language agreement rules (rarely needed in practice) |
| Type 0: Recursively enumerable | Turing machine (unbounded random-access memory) | Any computable language |

Each class strictly contains the one above it — every regular language is context-free, but not
vice versa — because more memory can only add recognizing power, never remove it.

### Why this matters for generation, not just recognition

Everything above describes *recognizing* whether a string belongs to a language. The same
machinery runs in reverse for *generating* only strings that belong to a language: at each step,
restrict the available next symbols to whichever ones keep the automaton (or grammar) in a state
from which acceptance is still reachable. This "generate only what the automaton permits" idea
is exactly **constrained decoding**, developed in the AI section below.

---

In [ ]:
# Setup
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A DFA's state-transition diagram makes "no memory beyond the current state" visible directly:
every state is a single node, and every transition is an edge labeled by the symbol that
triggers it. We draw the DFA this notebook builds from scratch below, using `networkx` rather
than `graphviz` rendering (no external `dot` binary needed).

In [ ]:
# State-transition diagram for a DFA recognizing balanced parentheses UP TO a fixed max depth.
# (Unbounded-depth balanced parentheses is provably NOT regular -- see the Theory section's
# pumping-lemma-style argument -- so this DFA deliberately caps depth to stay within what a
# finite number of states can track.)
MAX_DEPTH = 3
G = nx.DiGraph()
nodes = list(range(MAX_DEPTH + 1)) + ["REJECT"]
G.add_nodes_from(nodes)

edge_labels = {}
for state in range(MAX_DEPTH + 1):
    open_target = state + 1 if state < MAX_DEPTH else "REJECT"
    G.add_edge(state, open_target)
    edge_labels[(state, open_target)] = edge_labels.get((state, open_target), "") + "("

    close_target = state - 1 if state > 0 else "REJECT"
    G.add_edge(state, close_target)
    edge_labels[(state, close_target)] = edge_labels.get((state, close_target), "") + ")"
G.add_edge("REJECT", "REJECT")
edge_labels[("REJECT", "REJECT")] = "(, )"

pos = {0: (0, 0), 1: (3, 0), 2: (6, 0), 3: (9, 0), "REJECT": (4.5, -3)}
fig, ax = plt.subplots(figsize=(11, 6))
node_colors = ["#55A868" if n == 0 else "#C44E52" if n == "REJECT" else "#4C72B0" for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, ax=ax)
labels = {0: "depth 0\n(start+accept)", 1: "depth 1", 2: "depth 2",
          3: "depth 3 (max)", "REJECT": "REJECT"}
nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color="white", ax=ax)
for u, v in G.edges():
    if u != v:
        nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], connectionstyle="arc3,rad=0.2",
                                 ax=ax, arrowsize=18, node_size=1800)
for (u, v), label in edge_labels.items():
    if u == v:
        continue
    x, y = (pos[u][0] + pos[v][0]) / 2, (pos[u][1] + pos[v][1]) / 2
    y += 0.4 if (isinstance(u, int) and isinstance(v, int) and v > u) else -0.4
    ax.annotate(label, (x, y), fontsize=11, ha="center", color="#333333")
ax.set_title(f"DFA recognizing balanced parentheses up to depth {MAX_DEPTH}\n"
              "(state = current nesting depth; states beyond the cap all merge into REJECT)")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"States: {MAX_DEPTH + 2} total (depths 0 through {MAX_DEPTH}, plus REJECT)")
print("Start state = depth 0. Accepting state = depth 0 (a string is balanced only if every")
print("open paren was eventually closed, ending back at zero net depth).")

## 🐍 Implementation from Scratch

We implement a DFA as an explicit state-transition table (not just a description of one), run
it on a batch of test strings, and cross-check every verdict against an independent
stack-based balanced-parentheses checker written completely separately from the DFA's own
transition logic — the ground truth the DFA's design is trying to match, within its depth cap.

In [ ]:
class DFA:
    """A deterministic finite automaton, as literally as the Theory section's definition:
    a transition TABLE, a start state, and a set of accepting states. No hidden extra memory."""
    def __init__(self, alphabet, transition, start, accepting):
        self.alphabet = alphabet
        self.transition = transition  # dict: (state, symbol) -> next_state
        self.start = start
        self.accepting = accepting

    def run(self, string):
        """Feed the string one symbol at a time; accept iff we END in an accepting state."""
        state = self.start
        for ch in string:
            if ch not in self.alphabet:
                return False
            state = self.transition[(state, ch)]
        return state in self.accepting

def build_bounded_parens_dfa(max_depth):
    """States 0..max_depth track current nesting depth; REJECT is a trap state for anything
    that goes negative (unmatched close) or over the cap (deeper than max_depth allows)."""
    transition = {}
    for depth in range(max_depth + 1):
        transition[(depth, "(")] = depth + 1 if depth < max_depth else "REJECT"
        transition[(depth, ")")] = depth - 1 if depth > 0 else "REJECT"
    transition[("REJECT", "(")] = "REJECT"
    transition[("REJECT", ")")] = "REJECT"
    return DFA(alphabet={"(", ")"}, transition=transition, start=0, accepting={0})

MAX_DEPTH = 3
paren_dfa = build_bounded_parens_dfa(MAX_DEPTH)

def stack_based_checker(s, max_depth=MAX_DEPTH):
    """An INDEPENDENT implementation (a running counter, no shared code with the DFA above)
    to check the DFA's verdicts against -- the DFA could have a real transition-table bug."""
    depth = 0
    for ch in s:
        if ch == "(":
            depth += 1
            if depth > max_depth:
                return False
        elif ch == ")":
            depth -= 1
            if depth < 0:
                return False
        else:
            return False
    return depth == 0

test_strings = ["", "()", "(())", "((()))", "(((())))", ")(", "(()",
                 "()()()", "((()()))", "((((()))))", "(()(()))"]
print(f"{'string':<16} {'DFA accepts':>12} {'stack-checker':>14} {'match':>7}")
mismatches = 0
for s in test_strings:
    dfa_verdict = paren_dfa.run(s)
    checker_verdict = stack_based_checker(s)
    if dfa_verdict != checker_verdict:
        mismatches += 1
    print(f"{s!r:<16} {dfa_verdict!s:>12} {checker_verdict!s:>14} {dfa_verdict == checker_verdict!s:>7}")
print(f"\nMismatches across {len(test_strings)} test strings: {mismatches}")
print(f"Note '(((())))' (depth 4) is correctly REJECTED even though it's validly balanced --")
print(f"this DFA only recognizes balanced strings up to depth {MAX_DEPTH}, by construction.")

## 🤖 Why This Matters for AI

An LLM sampling tokens with no constraints will produce grammatically or syntactically invalid
output at some nonzero rate — nothing in a raw softmax distribution over the vocabulary knows
about parenthesis balance, valid JSON syntax, or a regular expression's structure.
**Constrained decoding** fixes this by tracking an automaton's state alongside generation and
**masking out** (zeroing the probability of) any next token that would move the automaton into
a state from which the language can no longer be completed validly — exactly the DFA's
transition table telling you, at every step, which symbols are even legal to consider. This is
the mechanism behind tools like `outlines`, `guidance`, and grammar-constrained decoding in
`llama.cpp`: the grammar compiles down to something automaton-like, and every generation step
consults it before sampling. Below, we force a toy token-level "language model" to only ever
generate strings inside our bounded-depth-parentheses DFA's language.

In [ ]:
vocab = ["(", ")", "END"]  # a toy 3-token "language model" vocabulary

def allowed_next_tokens(state):
    """Ask the DFA which tokens are even legal from the current state -- exactly the mask a
    grammar-constrained decoder applies to a real LLM's next-token distribution."""
    allowed = []
    for token in vocab:
        if token == "END":
            if state in paren_dfa.accepting:  # only allowed to stop on a BALANCED string
                allowed.append(token)
        else:
            next_state = paren_dfa.transition.get((state, token))
            if next_state is not None and next_state != "REJECT":
                allowed.append(token)
    return allowed

def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def generate(rng, constrained, max_len=40):
    """A toy 'language model': random logits at every step stand in for a real model's
    next-token scores. `constrained=True` masks those scores through the DFA before sampling."""
    state, generated = 0, []
    for _ in range(max_len):
        logits = rng.normal(size=len(vocab))       # stand-in for a real model's next-token logits
        probs = softmax(logits)
        if constrained:
            mask = np.array([1.0 if tok in allowed_next_tokens(state) else 0.0 for tok in vocab])
            probs = probs * mask
            probs = probs / probs.sum()              # renormalize over just the legal tokens
        token = vocab[rng.choice(len(vocab), p=probs)]
        if token == "END":
            return "".join(generated), True
        generated.append(token)
        state = paren_dfa.transition[(state, token)]
        if state == "REJECT" and not constrained:
            return "".join(generated) + "<INVALID>", False
    return "".join(generated), False

print("Sample UNCONSTRAINED generations (random logits, no DFA masking):")
for seed in range(5):
    text, _ = generate(np.random.default_rng(seed), constrained=False)
    print(f"  {text!r:20} valid balanced string: {paren_dfa.run(text.replace('<INVALID>', ''))}")

print("\nSample CONSTRAINED generations (same random logits, masked through the DFA):")
for seed in range(5):
    text, _ = generate(np.random.default_rng(seed + 100_000), constrained=True)
    print(f"  {text!r:20} valid balanced string: {paren_dfa.run(text)}")

# Measure the effect over many trials, using FRESH seeds disjoint from the 10 shown above.
# IMPORTANT: the empty string trivially counts as "balanced" in both conditions, which would
# inflate BOTH percentages for a reason that has nothing to do with constraining -- so we
# additionally report the non-empty-only rate, which isolates the actual effect of the mask.
N_TRIALS = 300
unconstrained_valid = constrained_valid = constrained_ended_but_invalid = 0
u_nonempty_attempts = u_nonempty_valid = c_nonempty_attempts = c_nonempty_valid = 0
for seed in range(N_TRIALS):
    text_u, ended_u = generate(np.random.default_rng(seed + 200_000), constrained=False)
    clean_u = text_u.replace("<INVALID>", "")
    if ended_u and "<INVALID>" not in text_u and paren_dfa.run(text_u):
        unconstrained_valid += 1
    if len(clean_u) > 0:
        u_nonempty_attempts += 1
        if ended_u and paren_dfa.run(clean_u):
            u_nonempty_valid += 1

    text_c, ended_c = generate(np.random.default_rng(seed + 300_000), constrained=True)
    if paren_dfa.run(text_c):
        constrained_valid += 1
    elif ended_c:
        constrained_ended_but_invalid += 1  # should NEVER happen if the mask is built correctly
    if len(text_c) > 0:
        c_nonempty_attempts += 1
        if paren_dfa.run(text_c):
            c_nonempty_valid += 1

print(f"\nOver {N_TRIALS} trials (including trivially-valid empty strings):")
print(f"  Unconstrained: {unconstrained_valid}/{N_TRIALS} = {100*unconstrained_valid/N_TRIALS:.1f}% valid")
print(f"  Constrained:   {constrained_valid}/{N_TRIALS} = {100*constrained_valid/N_TRIALS:.1f}% valid")
print(f"  Constrained generations that ended but were STILL invalid: {constrained_ended_but_invalid} "
      f"(confirms the mask, not luck, is producing validity)")

print(f"\nRestricting to only the NON-EMPTY generations (the empty string is a trivial case that")
print(f"would be 'valid' either way, so it can't be evidence the mask is doing anything):")
print(f"  Unconstrained non-empty: {u_nonempty_valid}/{u_nonempty_attempts} = "
      f"{100*u_nonempty_valid/u_nonempty_attempts:.1f}% valid")
print(f"  Constrained non-empty:   {c_nonempty_valid}/{c_nonempty_attempts} = "
      f"{100*c_nonempty_valid/c_nonempty_attempts:.1f}% valid")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, trace `paren_dfa`'s state after each character of `"(()"` and confirm it ends in a
   non-accepting state (so the DFA correctly rejects it). Then trace `"(())"` and confirm it
   ends in state 0 (accepting).

<details>
<summary>💡 Solution</summary>

Starting at state 0: `"(()"` visits $0\to1\to2\to1$ (open, open, close), ending at state 1 —
not in `accepting={0}`, so correctly rejected (there's one unmatched open paren). `"(())"`
visits $0\to1\to2\to1\to0$ (open, open, close, close), ending at state 0 — accepted, matching
the fact that the string is genuinely balanced.

</details>

### 💭 UNDERSTAND
2. `paren_dfa` (built with `MAX_DEPTH=3`) correctly rejects `"(((())))"`, a string that IS
   validly balanced (every open paren has a matching close). Explain why this is not a bug in
   the DFA's transition table, but an unavoidable consequence of capping nesting depth at a
   fixed number of states — and state exactly what would need to change about the DFA (not
   about the string) to accept this particular string while still rejecting truly unbalanced
   ones.

<details>
<summary>💡 Solution</summary>

`"(((())))"` reaches nesting depth 4 at its midpoint, but `paren_dfa` only has states for
depths 0 through 3 — the moment depth would need to reach 4, the transition table sends it to
`REJECT`, a trap state with no way back. This isn't a table bug; it's the DFA correctly
recognizing exactly the language it was built for ("balanced up to depth 3"), which is a
strict subset of "balanced." To accept `"(((())))"` while still rejecting genuinely unbalanced
strings, you'd need to rebuild the DFA with `MAX_DEPTH=4` (or higher) — one more state added to
the transition table — not touch the string or the acceptance rule at all. This is exactly the
Theory section's point made concrete: no *fixed* number of states can handle *unbounded* depth,
so any DFA for this pattern always has some depth beyond which it starts rejecting valid
strings.

</details>

### ✏️ DERIVE
3. Derive the mod-3 DFA's transition rule used in the IMPLEMENT exercise below: for a binary
   string read most-significant-bit first, show that if the value read so far has remainder $r$
   mod 3, then appending bit $b$ produces a new value with remainder $(2r+b) \bmod 3$.

<details>
<summary>💡 Solution outline</summary>

If the bits read so far represent integer $v$ with $v \equiv r \pmod 3$, appending bit $b$
(most-significant-bit-first reading) produces the new value $v' = 2v + b$ — shifting every
existing bit one place left (doubling the value) and adding the new bit. Taking both sides mod
3: $v' \bmod 3 = (2v+b) \bmod 3$. Since modular arithmetic respects multiplication and addition
($2v \bmod 3$ depends only on $v \bmod 3 = r$, i.e. $2v \equiv 2r \pmod 3$), this becomes
$v' \bmod 3 = (2r+b) \bmod 3$ — exactly the claimed state-transition rule, derived from the
definition of positional binary notation rather than assumed.

</details>

### 🐍 IMPLEMENT
4. Build a new DFA (reusing the `DFA` class) that accepts binary strings representing a number
   divisible by 3, reading most-significant-bit first, using the transition rule derived above:
   from remainder-state $r$, reading bit $b$ moves to state $(2r+b) \bmod 3$.

<details>
<summary>✅ How to know you're right</summary>

Testing on binary representations of several numbers divisible by 3 (6→`"110"`, 12→`"1100"`,
9→`"1001"`, 18→`"10010"`) should all return `True` from your DFA's `.run()`, while several
numbers NOT divisible by 3 (5→`"101"`, 7→`"111"`, 10→`"1010"`, 11→`"1011"`) should all return
`False` — cross-checked directly against `int(bitstring, 2) % 3 == 0` for every test case, not
just eyeballed.

</details>

### 🤖 APPLY
5. The notebook's constrained-decoding demo masks a toy LM's next-token distribution using
   `paren_dfa`. Adapt `allowed_next_tokens` and `generate` to instead constrain generation using
   your mod-3 DFA from IMPLEMENT — i.e. build a toy "language model" over the vocabulary
   `["0", "1", "END"]` that only ever generates binary strings representing multiples of 3.

<details>
<summary>✅ What you should observe</summary>

Exactly as the notebook's own parentheses demo showed, unconstrained generation should produce
multiples of 3 only by chance (roughly 1/3 of the time among strings that happen to end validly,
since a uniformly random bitstring is divisible by 3 with probability near 1/3), while every
single constrained generation should satisfy `int(bitstring, 2) % 3 == 0` — because `END` is
only ever unmasked when the DFA's current state is the accepting remainder-0 state, the same
"only allowed to stop on acceptance" logic `allowed_next_tokens` already uses for the
parentheses DFA.

</details>

### 🚀 CHALLENGE
6. The constrained-decoding demo only ever produces syntactically balanced parentheses, but
   says nothing about them being *useful*. Extend `allowed_next_tokens` and `paren_dfa`'s
   vocabulary to include digits `0`-`9` and `+`, with a grammar restricted to expressions like
   `(1+2)` — balanced parens wrapping simple sums (you'll need more DFA states to track "just
   saw a digit" vs. "just saw an operator" vs. "inside/outside parens"). Generate 20 constrained
   samples and report what fraction are not just syntactically valid but also evaluate to a
   correct arithmetic result if you `eval()` them — this distinguishes what grammar-constrained
   decoding *guarantees* (syntactic validity) from what it does *not* (semantic correctness).

<details>
<summary>🔍 What a strong answer covers</summary>

A correct grammar extension tracks enough state to know, at every point, exactly which of
`(`, digits, `+`, `)`, `END` are syntactically legal next (e.g. right after `(`, only a digit is
legal; right after a digit, only `+` or `)` is legal; right after `+`, only a digit is legal).
Every generated sample should be syntactically valid by construction (the mask guarantees this,
the same way it guaranteed balanced parens). But `eval()`-ing them will reveal that *evaluating
to a correct result* isn't the same claim at all — every sample trivially "evaluates correctly"
in the sense that Python's `+` operator faithfully computes whatever sum was generated (e.g.
`(3+7)` evaluates to `10`, which is simply correct arithmetic, not a coincidence), so a strong
answer recognizes that *this particular grammar* can't actually generate an arithmetically wrong
expression — the interesting failure mode only shows up if the grammar is extended further to
allow expressions that could be syntactically valid but semantically meaningless in a richer
sense (e.g. division by a generated zero, or nonsensical results in a grammar for a typed
language where syntax doesn't imply type-correctness). The core takeaway a strong answer should
state explicitly: grammar-constrained decoding's guarantee is scoped exactly to what the
grammar encodes, and "syntactically valid" and "semantically/logically correct" are genuinely
different properties that happen to coincide for simple arithmetic but diverge for most
real-world structured-generation tasks (valid-but-wrong JSON values, syntactically valid-but-
buggy code, etc.).

---

## 📚 Further Reading
- Sipser, *Introduction to the Theory of Computation*, Ch. 1-2 (Finite Automata, Context-Free Grammars)
- Willard, ["Efficient Guided Generation for Large Language Models"](https://arxiv.org/abs/2307.09702) (the `outlines` library's approach to grammar-constrained decoding)
- Geng et al., ["Grammar-Constrained Decoding for Structured NLP Tasks without Finetuning"](https://arxiv.org/abs/2305.13971)

---

*Next notebook: [Discrete Math in NLP](05_discrete_math_in_nlp.ipynb)*